# Lab 2.1 (ต่อ) · Data Profiling ข้อมูลลูกค้า customers.csv

ต่อจากข้อ **🚀 เสร็จเร็ว? ลองต่อ**: *ลองอ่าน customers.csv แล้วทำ profiling ด้วย prompt เดิม มีปัญหาอะไรไหม*

ข้อมูล customers.csv คือรายชื่อสมาชิกร้านบ้านบรู 1 แถว = ลูกค้า 1 คน เป้าหมายคือ **หาความผิดปกติก่อน ยังไม่แก้** แล้วเทียบกับ sales.csv และ branches.csv ว่าข้อมูลสามตารางเชื่อมกันได้ถูกต้องไหม

| ขั้น | ทำอะไร |
|---|---|
| 1 | โหลดและดูข้อมูลครั้งแรก |
| 2 | Profiling ตารางเดียว (prompt เดิม ปรับคอลัมน์ให้ตรง) |
| 3 | ตรวจข้ามตาราง: customers ↔ sales ↔ branches |
| 4 | สรุปปัญหาและตัดสินใจ |
| 5 | Export ไฟล์ที่เพิ่มคอลัมน์ช่วยวิเคราะห์ (ไม่ลบแถว) |

**ไฟล์ที่ต้องอัปโหลด** (อยู่ในโฟลเดอร์ public/ ของโปรเจกต์ Dashboard): customers.csv, sales.csv, branches.csv

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้) มีฟังก์ชัน load_csv() และ check() สำหรับตรวจคำตอบโดยไม่เฉลยตัวเลข เหมือน Lab 2.1

In [ ]:
# ===== เซลล์ตัวช่วย · รันก่อน ไม่ต้องแก้ =====
import base64, json, os, re
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

_EXP = json.loads(base64.b64decode("eyJyb3dzIjozMDAwLCJkdXBfcm93cyI6MCwiZHVwX2lkIjowLCJibGFua19jZWxscyI6MCwiZ2VuZGVyX3Vuc3BlY2lmaWVkIjoxMTYsInBob25lX3NoYXJlZCI6OTksInVua25vd25fY3VzdG9tZXIiOjAsIm5ldmVyX2JvdWdodCI6NDkzLCJidXlfYmVmb3JlX2pvaW4iOjB9").decode())
_LABEL = {
    "rows": "จำนวนลูกค้าทั้งหมด",
    "dup_rows": "แถวที่ซ้ำกันทุกคอลัมน์",
    "dup_id": "customer_id ที่ซ้ำ",
    "blank_cells": "จำนวนช่องว่างรวมทุกคอลัมน์",
    "gender_unspecified": "gender = 'ไม่ระบุ'",
    "phone_shared": "ลูกค้าที่เบอร์โทรชนกับคนอื่น (นับทุกคนในกลุ่ม)",
    "unknown_customer": "customer_id ใน sales ที่ไม่มีใน customers",
    "never_bought": "สมาชิกที่ไม่เคยซื้อเลย",
    "buy_before_join": "สมาชิกที่ซื้อครั้งแรกก่อนวันสมัคร",
}
_HINT = {
    "blank_cells": "อ่านไฟล์ด้วย keep_default_na=False ค่าว่างจึงเป็น '' ลอง (df.apply(lambda s: s.str.strip()) == '').sum()",
    "gender_unspecified": "'ไม่ระบุ' ไม่ใช่ค่าว่าง ต้องนับจาก value_counts()",
    "phone_shared": "ใช้ duplicated('phone', keep=False) จะนับทุกคนที่เบอร์ชนกัน ไม่ใช่แค่คนที่สอง",
    "unknown_customer": "ตัดแถวที่ customer_id ว่างใน sales ออกก่อน (ลูกค้าไม่ใช่สมาชิก)",
    "never_bought": "เอา set ของ customer_id ใน customers ลบด้วย set ใน sales",
    "buy_before_join": "ใช้วันแรกที่ซื้อของแต่ละคน (groupby แล้ว min) เทียบกับ joined_date",
}


def check(name, value):
    """เทียบคำตอบกับเฉลยโดยไม่บอกตัวเลข"""
    exp = _EXP[name]
    label = _LABEL[name]
    if value is None:
        print(f"⬜ {label}: ยังไม่ได้ใส่ค่า")
    elif int(value) == exp:
        print(f"✅ {label}: {int(value):,}")
    elif abs(int(value) - exp) <= max(2, exp * 0.03):
        print(f"🟡 {label}: {int(value):,} ใกล้แล้ว แต่ยังไม่ตรง · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")
    else:
        print(f"❌ {label}: {int(value):,} ยังไม่ถูก · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")


NEEDED = ["customers.csv", "sales.csv", "branches.csv"]


def upload_needed():
    """ให้อัปโหลดไฟล์ที่ยังขาด เลือกหลายไฟล์พร้อมกันได้ (กด Ctrl ค้างแล้วคลิก)"""
    missing = [n for n in NEEDED if not os.path.exists(n)]
    if not missing:
        print("✅ มีไฟล์ครบแล้ว:", ", ".join(NEEDED))
        return
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"ไม่พบ {missing} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    print("กรุณาเลือกไฟล์เหล่านี้ (เลือกพร้อมกันได้):", ", ".join(missing))
    for got in files.upload():
        # Colab ตั้งชื่อใหม่เป็น "sales (1).csv" ถ้าอัปซ้ำ → เปลี่ยนกลับเป็นชื่อเดิม
        base = re.sub(r" \(\d+\)(?=\.csv$)", "", got)
        if base in NEEDED and got != base:
            os.replace(got, base)
    still = [n for n in NEEDED if not os.path.exists(n)]
    if still:
        print("❌ ยังขาด:", ", ".join(still), "· รันเซลล์นี้อีกครั้งแล้วเลือกไฟล์ที่ขาด")
    else:
        print("✅ มีไฟล์ครบแล้ว:", ", ".join(NEEDED))


def load_csv(name):
    """อ่าน CSV ทุกคอลัมน์เป็นข้อความ"""
    if not os.path.exists(name):
        raise FileNotFoundError(f"ไม่พบ {name} · รันเซลล์ upload_needed() ด้านบนแล้วเลือกไฟล์นี้")
    return pd.read_csv(name, dtype=str, keep_default_na=False)

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก

**1.1 อัปโหลดไฟล์** รันเซลล์ถัดไป แล้วกด "Choose Files" เลือก customers.csv, sales.csv, branches.csv **พร้อมกันทั้ง 3 ไฟล์** (กด Ctrl ค้างแล้วคลิก) ไฟล์อยู่ในโฟลเดอร์ public/ ของโปรเจกต์ Dashboard

> Colab รันบนเครื่องของ Google มองไม่เห็นไฟล์ในคอมเรา ต้องอัปโหลดทุกครั้งที่เปิด notebook ใหม่ (หรือหลัง Runtime ถูกรีเซ็ต)

In [ ]:
upload_needed()

**1.2 อ่านข้อมูล** อ่าน **ทุกคอลัมน์เป็นข้อความ** เหมือนเดิม เพื่อไม่ให้ pandas ซ่อนปัญหา (เช่น ค่าว่างกลายเป็น NaN หรือวันที่ที่ผิดรูปแบบถูกแปลงเงียบ ๆ)

In [ ]:
cust = load_csv("customers.csv")
sales = load_csv("sales.csv")
branches = load_csv("branches.csv")

print(f"customers: {len(cust):,} แถว × {cust.shape[1]} คอลัมน์ → {list(cust.columns)}")
print(f"sales: {len(sales):,} แถว · branches: {len(branches)} สาขา")
cust.head(8)

**คำถามชวนคิด:** คอลัมน์ไหนน่าจะเป็น "กุญแจ" (key) ที่ใช้เชื่อมกับ sales.csv และคอลัมน์ไหนเชื่อมกับ branches.csv ชื่อคอลัมน์ตรงกันไหม

## ขั้นที่ 2 · Profiling ตารางเดียว: หาปัญหาก่อน ยังไม่แก้

**Prompt** (ดัดแปลงจาก prompt เดิมของ Lab 2.1 ให้ตรงกับคอลัมน์ของ customers)
```
ช่วยทำ data profiling ของ DataFrame cust ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
คอลัมน์: customer_id, nickname, gender, age_group, home_branch_id, joined_date, phone
- จำนวนค่าว่าง ค่าไม่ซ้ำ และค่าที่มีช่องว่างหัว/ท้าย ของแต่ละคอลัมน์
- จำนวนแถวที่ซ้ำกันทุกคอลัมน์ และ customer_id ที่ซ้ำ / ผิดรูปแบบ C + ตัวเลข 5 หลัก
- ค่าทั้งหมดพร้อมจำนวนแถวของ gender, age_group, home_branch_id
- joined_date: กี่แถวเป็น YYYY-MM-DD กี่แถวเป็นเลข serial ของ Excel ช่วงวันที่ และจำนวนสมัครรายเดือน
- phone: รูปแบบของเบอร์ (แทนตัวเลขด้วย 9) และจำนวนลูกค้าที่เบอร์ชนกัน พร้อมตัวอย่าง
- nickname มีกี่ชื่อไม่ซ้ำ ใช้ระบุตัวคนได้ไหม
เก็บจำนวนไว้ในตัวแปร n_blank, n_dup, n_dup_id, n_gender_unspecified, n_phone_shared
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

In [ ]:
# ===== ขั้นที่ 2 · Data profiling ของ customers (รายงานอย่างเดียว ไม่แก้ cust) =====
print(f"จำนวนแถวทั้งหมด: {len(cust):,}\n")

# 1) ค่าว่าง / ค่าไม่ซ้ำ / ช่องว่างหัว-ท้าย ต่อคอลัมน์
overview = pd.DataFrame({
    "ค่าว่าง": (cust.apply(lambda s: s.str.strip()) == "").sum(),
    "ค่าไม่ซ้ำ": cust.nunique(),
    "มีช่องว่างหัว/ท้าย": cust.apply(lambda s: s != s.str.strip()).sum(),
})
print("1) ภาพรวมต่อคอลัมน์")
print(overview, "\n")
n_blank = int(overview["ค่าว่าง"].sum())

# 2) แถวซ้ำ และ customer_id
n_dup = int(cust.duplicated().sum())
n_dup_id = int(cust["customer_id"].duplicated().sum())
bad_id = ~cust["customer_id"].str.match(r"^C\d{5}$")
print(f"2) แถวซ้ำทุกคอลัมน์: {n_dup:,} · customer_id ซ้ำ: {n_dup_id:,} · customer_id ผิดรูปแบบ: {int(bad_id.sum()):,}\n")

# 3) ค่าหมวดหมู่ (ใช้ repr ให้เห็นช่องว่างที่ซ่อนอยู่)
for col in ["gender", "age_group", "home_branch_id"]:
    counts = cust[col].value_counts()
    counts.index = [repr(v) for v in counts.index]
    print(f"3) {col}")
    print(counts.to_string(), "\n")
n_gender_unspecified = int((cust["gender"] == "ไม่ระบุ").sum())

# 4) joined_date
is_iso = cust["joined_date"].str.match(r"^\d{4}-\d{2}-\d{2}$")
is_serial = cust["joined_date"].str.fullmatch(r"\d{5}")
print(f"4) joined_date เป็น YYYY-MM-DD: {int(is_iso.sum()):,} · เป็นเลข serial ของ Excel (เช่น 46084): {int(is_serial.sum()):,} · อื่น ๆ: {int((~is_iso & ~is_serial).sum()):,}")
joined = pd.to_datetime(cust["joined_date"].where(is_iso), format="%Y-%m-%d", errors="coerce")
print(f"   ช่วงวันที่สมัคร: {joined.min():%Y-%m-%d} ถึง {joined.max():%Y-%m-%d}")
print("   จำนวนสมัครรายเดือน:")
print(joined.dt.to_period("M").value_counts().sort_index().to_string(), "\n")

# 5) phone
print("5) รูปแบบเบอร์โทร")
print(cust["phone"].str.replace(r"\d", "9", regex=True).value_counts().to_string())
shared = cust["phone"].duplicated(keep=False)
n_phone_shared = int(shared.sum())
print(f"ลูกค้าที่เบอร์ชนกับคนอื่น: {n_phone_shared:,} คน ({cust.loc[shared, 'phone'].nunique():,} เบอร์)")
print(cust[shared].sort_values("phone").head(6).to_string(index=False), "\n")

# 6) nickname
print(f"6) nickname มี {cust['nickname'].nunique()} ชื่อไม่ซ้ำ จากลูกค้า {len(cust):,} คน")
print(cust["nickname"].value_counts().head(5).to_string())

### ✔️ ตรวจคำตอบ

In [ ]:
check("rows", len(cust))
check("dup_rows", n_dup)
check("dup_id", n_dup_id)
check("blank_cells", n_blank)
check("gender_unspecified", n_gender_unspecified)
check("phone_shared", n_phone_shared)

## ขั้นที่ 3 · ตรวจข้ามตาราง (customers ↔ sales ↔ branches)

ตารางเดียวดูสะอาด **ไม่ได้แปลว่าใช้งานได้** ต้องดูว่าเชื่อมกับตารางอื่นถูกไหม

**Prompt**
```
มี DataFrame cust (ลูกค้า), sales (ยอดขาย 1 แถว = 1 รายการ, customer_id ว่าง = ไม่ใช่สมาชิก)
และ branches (branch_id, branch, opened_date) ทุกคอลัมน์เป็น str
ช่วยตรวจความสอดคล้องระหว่างตาราง โดยไม่แก้ข้อมูล:
- customer_id ใน sales ที่ไม่มีใน cust และสมาชิกใน cust ที่ไม่เคยซื้อเลย
- home_branch_id ที่ไม่มีใน branches และลูกค้าที่สมัครก่อนสาขาประจำเปิด
- ลูกค้าที่ซื้อครั้งแรก (วันที่ = 10 ตัวแรกของ datetime) ก่อน joined_date
- สาขาประจำ (map home_branch_id เป็นชื่อ) ต่างจากสาขาที่ซื้อบ่อยที่สุดกี่คน
- ช่วงวันที่ของ sales เทียบกับช่วงวันที่สมัคร
เก็บจำนวนไว้ในตัวแปร n_unknown_customer, n_never_bought, n_buy_before_join
```

In [ ]:
# ===== ขั้นที่ 3 · ตรวจข้ามตาราง (รายงานอย่างเดียว) =====
if not cust["joined_date"].str.match(r"^\d{4}-\d{2}-\d{2}$").all():
    print("⚠️ joined_date ยังไม่เป็น YYYY-MM-DD ทุกแถว การเทียบวันที่ด้านล่างจะเชื่อไม่ได้\n")

ids = set(cust["customer_id"])
member_sales = sales[sales["customer_id"].str.strip() != ""]
sale_ids = set(member_sales["customer_id"])
n_unknown_customer = len(sale_ids - ids)
n_never_bought = len(ids - sale_ids)
print(f"1) สมาชิกที่มีใน sales: {len(sale_ids):,} คน · ไม่มีใน customers: {n_unknown_customer:,}")
print(f"   สมาชิกที่ไม่เคยซื้อเลย: {n_never_bought:,} คน ({n_never_bought / len(ids):.1%})\n")

# 2) home_branch_id ↔ branches
b = branches.set_index("branch_id")
n_bad_home = int((~cust["home_branch_id"].isin(b.index)).sum())
opened = cust["home_branch_id"].map(b["opened_date"])
n_before_open = int((cust["joined_date"] < opened).sum())  # เทียบเป็นข้อความได้เพราะเป็น YYYY-MM-DD ทั้งคู่
print(f"2) home_branch_id ที่ไม่มีใน branches: {n_bad_home:,} · สมัครก่อนสาขาประจำเปิด: {n_before_open:,}\n")

# 3) ซื้อครั้งแรกก่อนวันสมัคร
first_buy = member_sales["datetime"].str[:10].groupby(member_sales["customer_id"]).min()
joined_by_id = cust.set_index("customer_id")["joined_date"]
both = pd.DataFrame({"first_buy": first_buy, "joined_date": joined_by_id}).dropna()
n_buy_before_join = int((both["first_buy"] < both["joined_date"]).sum())
print(f"3) ซื้อครั้งแรกก่อนวันสมัคร: {n_buy_before_join:,} คน\n")

# 4) สาขาประจำ vs สาขาที่ซื้อบ่อยที่สุด
top_branch = (member_sales.groupby(["customer_id", "branch"]).size()
              .reset_index(name="n")
              .sort_values(["customer_id", "n"], ascending=[True, False])
              .drop_duplicates("customer_id")
              .set_index("customer_id")["branch"])
home = cust.set_index("customer_id")["home_branch_id"].map(b["branch"])
cmp = pd.DataFrame({"home": home, "top": top_branch}).dropna()
n_home_mismatch = int((cmp["home"] != cmp["top"]).sum())
print(f"4) สาขาประจำไม่ตรงกับสาขาที่ซื้อบ่อยที่สุด: {n_home_mismatch:,} จาก {len(cmp):,} คน ({n_home_mismatch / len(cmp):.1%})\n")

# 5) ช่วงเวลาของข้อมูล
sale_dates = sales["datetime"].str[:10]
print(f"5) sales: {sale_dates.min()} ถึง {sale_dates.max()}")
print(f"   สมัครสมาชิก: {cust['joined_date'].min()} ถึง {cust['joined_date'].max()}")
print("   วันเปิดสาขา:")
print(branches[["branch_id", "branch", "opened_date"]].to_string(index=False))

In [ ]:
check("unknown_customer", n_unknown_customer)
check("never_bought", n_never_bought)
check("buy_before_join", n_buy_before_join)

In [ ]:
# จำนวนสมาชิกใหม่รายเดือน (ป้ายภาษาอังกฤษ เพราะ matplotlib ใน Colab ไม่มีฟอนต์ไทย)
import matplotlib.pyplot as plt
monthly = pd.to_datetime(cust["joined_date"], format="%Y-%m-%d").dt.to_period("M").value_counts().sort_index()
ax = monthly.plot.bar(color="#0F8B8D", figsize=(10, 3.5))
ax.set_xlabel("Joined month"); ax.set_ylabel("New members")
ax.set_title("New members per month (last month may be partial)")
plt.tight_layout(); plt.show()

## ขั้นที่ 4 · สรุปปัญหาและตัดสินใจ

**คำตอบของ "มีปัญหาอะไรไหม":** ตารางนี้ *สะอาดกว่า sales_raw มาก* ไม่มีค่าว่าง แถวซ้ำ id ซ้ำ หรือช่องว่างท้ายค่า แต่มีเรื่องที่ต้องระวังเวลาใช้งาน ดับเบิลคลิกเซลล์นี้เพื่อแก้คอลัมน์ "ตัดสินใจ" ให้เป็นของคุณ

| ปัญหาที่พบ | ทำไมเป็นปัญหา | ทางเลือก | ตัดสินใจ (ตัวอย่าง) |
|---|---|---|---|
| joined_date เป็นเลข serial (เช่น 46084) ถ้าแปลงจาก .xlsx ไม่ถูกวิธี | เทียบ/เรียงวันที่ไม่ได้ | แปลงเป็น YYYY-MM-DD | แปลง (ไฟล์ที่ให้มาแปลงแล้ว) |
| home_branch_id เป็นรหัส B01–B05 แต่ sales ใช้ชื่อสาขา | join ตรง ๆ ไม่ได้ | map ผ่าน branches.csv | เพิ่มคอลัมน์ home_branch |
| เบอร์โทรถูกปิดบางส่วน (08x-xxx-1234) ทำให้ชนกัน 99 คน | ใช้ระบุตัวคน / หาลูกค้าซ้ำไม่ได้ | ใช้เป็น key / ไม่ใช้ | **ไม่ใช้ phone เป็น key** ใช้ customer_id |
| nickname มีแค่ 24 ชื่อ | ไม่ใช่ตัวระบุ ห้ามใช้หาคนซ้ำ | — | ใช้แสดงผลเท่านั้น |
| gender = 'ไม่ระบุ' 116 คน | ไม่ใช่ค่าว่าง แต่เป็นคำตอบที่ถูกต้อง | ลบ / เก็บ | เก็บไว้เป็นกลุ่มของตัวเอง |
| สมาชิก 493 คนไม่เคยซื้อ | ถ้าคำนวณ "ยอดต่อสมาชิก" ต้องเลือกว่าจะนับคนกลุ่มนี้ไหม | ลบ / เก็บ + flag | เก็บ และเพิ่ม flag has_purchase |
| สาขาประจำ ≠ สาขาที่ซื้อบ่อยสุด ประมาณ 5% | "สาขาประจำ" คือสาขาที่สมัคร ไม่ใช่สาขาที่ใช้จริง | — | ตีความให้ถูกตอนวิเคราะห์ |
| ลูกค้า "ต่ำกว่า 18" 113 คน | ข้อมูลผู้เยาว์ ต้องระวังเรื่องความยินยอมตาม PDPA | — | ไม่ใช้ทำการตลาดเจาะจงโดยไม่มีความยินยอม |
| วันสมัครเร็วสุดตรงกับวันแรกของ sales (1 เม.ย. 2568) แม้สาขาเปิดตั้งแต่ปี 2566 | ข้อมูลอาจถูกตัดช่วง ไม่ใช่สมาชิกทั้งหมดตั้งแต่เปิดร้าน | — | ระบุเป็นข้อจำกัดในรายงาน |

**สิ่งที่ตรวจแล้วผ่าน:** customer_id ใน sales มีใน customers ครบทุกคน · ไม่มีใครซื้อก่อนวันสมัคร · ไม่มีใครสมัครก่อนสาขาประจำเปิด

## ขั้นที่ 5 · Export

ไม่ลบแถวใด ๆ เพราะไม่พบข้อมูลผิด แต่เพิ่มคอลัมน์ช่วยวิเคราะห์ต่อท้าย (คอลัมน์เดิมยังอยู่ครบและเรียงเหมือนเดิม)
- home_branch ชื่อสาขาประจำ (ใช้ join กับ sales ได้)
- phone_shared เบอร์นี้ชนกับลูกค้าคนอื่นไหม
- has_purchase เคยซื้อไหม

In [ ]:
cust_out = cust.copy()
cust_out["home_branch"] = cust_out["home_branch_id"].map(b["branch"])
cust_out["phone_shared"] = cust_out["phone"].duplicated(keep=False)
cust_out["has_purchase"] = cust_out["customer_id"].isin(sale_ids)

profiling_log = pd.DataFrame([
    ("แถวทั้งหมด", len(cust), "—"),
    ("ค่าว่าง / แถวซ้ำ / id ซ้ำ", n_blank + n_dup + n_dup_id, "ไม่พบ ไม่ต้องแก้"),
    ("gender = ไม่ระบุ", n_gender_unspecified, "เก็บไว้เป็นกลุ่มของตัวเอง"),
    ("เบอร์โทรชนกัน (ถูกปิดบางส่วน)", n_phone_shared, "ไม่ใช้ phone เป็น key · เพิ่ม flag phone_shared"),
    ("customer_id ใน sales ที่ไม่มีใน customers", n_unknown_customer, "—"),
    ("สมาชิกที่ไม่เคยซื้อ", n_never_bought, "เก็บไว้ · เพิ่ม flag has_purchase"),
    ("ซื้อครั้งแรกก่อนวันสมัคร", n_buy_before_join, "—"),
    ("สาขาประจำ ≠ สาขาที่ซื้อบ่อยสุด", n_home_mismatch, "ตีความ home_branch ว่าเป็นสาขาที่สมัคร"),
], columns=["ขั้นตอน", "จำนวนแถวที่กระทบ", "การตัดสินใจ"])
display(profiling_log)

cust_out.to_csv("customers_profiled.csv", index=False, encoding="utf-8-sig")
profiling_log.to_csv("customers_profiling_log.csv", index=False, encoding="utf-8-sig")
print("บันทึก customers_profiled.csv และ customers_profiling_log.csv แล้ว")

try:
    from google.colab import files
    files.download("customers_profiled.csv")
    files.download("customers_profiling_log.csv")
except ImportError:
    pass

**คำถามชวนคิด:** ถ้าจะคำนวณ "ยอดซื้อเฉลี่ยต่อสมาชิก" ควรหารด้วยสมาชิกทั้งหมด หรือเฉพาะคนที่เคยซื้อ คำตอบต่างกันแค่ไหน และแบบไหนเหมาะกับคำถามธุรกิจอะไร